In [96]:
import torch
print(torch.__version__)

2.7.1+cu128


In [97]:
print(torch.cuda.is_available())

True


In [98]:
print(torch.tensor([1, 2, 3]))
print(torch.tensor([[1, 2, 3], [4, 5, 6]]))
print(torch.LongTensor([1, 2, 3]))
print(torch.FloatTensor([1, 2, 3]))


tensor([1, 2, 3])
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [99]:
tensor = torch.rand((3, 3), dtype = torch.float)
print(tensor)

tensor([[0.6195, 0.2391, 0.2689],
        [0.3315, 0.3122, 0.2912],
        [0.3652, 0.6299, 0.0954]])


In [100]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = torch.cuda.FloatTensor([1, 2, 3])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor = torch.rand((1, 1), device = device)
print(cpu)
print(gpu)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')


In [101]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to('cuda')

In [102]:
import numpy as np
ndarray = np.array([1, 2, 3], dtype = np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [103]:
tensor = torch.cuda.FloatTensor([1, 2, 3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(type(ndarray))


[1. 2. 3.]
<class 'numpy.ndarray'>


In [104]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split

In [105]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:, 1].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

In [106]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [107]:
dataset = CustomDataset('./dataset/non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
train_dataloader = DataLoader(train_dataset, batch_size = 16, shuffle = True, drop_last = True)
val_dataloader = DataLoader(val_dataset, batch_size = 4, shuffle = True, drop_last = True)
test_dataloader = DataLoader(test_dataset, batch_size = 4, shuffle = False, drop_last = True)

In [108]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [109]:
checkpoint = 0
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        torch.save(model, f'./models/checkpoint-{checkpoint}.pt')
        checkpoint += 1


100 tensor(0.0857, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.0821, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.0803, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.0809, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.0809, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.0812, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.0793, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.0790, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.0784, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.0779, device='cuda:0', grad_fn=<DivBackward0>)


In [110]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
    outputs = model(x)
    print(outputs)

tensor([[22.5025],
        [ 9.0274],
        [55.6473],
        [ 9.5594]], device='cuda:0')


In [111]:
torch.save(model, './models/model.pt')

In [112]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [113]:
model = torch.load('./models/model.pt', map_location = device, weights_only = False)

In [114]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [115]:
model_state_dict = torch.load('./models/model_state_dict.pt', map_location = device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [116]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [117]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.x3 = df.iloc[:, 2].values
        self.y = df.iloc[:, 3].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index], self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y

    def __len__(self):
        return self.length

In [118]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(3, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.layer(x)
        return x

In [119]:
dataset = CustomDataset('./dataset/binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], torch.manual_seed(42))
train_dataloader = DataLoader(train_dataset, batch_size = 16, shuffle = True, drop_last = True)
val_dataloader = DataLoader(val_dataset, batch_size = 4, shuffle = True, drop_last = True)
test_dataloader = DataLoader(test_dataset, batch_size = 4, shuffle = False, drop_last = True)

In [120]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [121]:
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        

100 tensor(0.6316, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.6289, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.6225, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.6150, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.6121, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.6033, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.6013, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.5980, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.5941, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.5868, device='cuda:0', grad_fn=<DivBackward0>)


In [122]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)
        print(outputs >= torch.FloatTensor([0.5]).to(device))

tensor([[0.5279],
        [0.6185],
        [0.5403],
        [0.5925]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.4804],
        [0.5484],
        [0.4598],
        [0.4383]], device='cuda:0')
tensor([[False],
        [ True],
        [False],
        [False]], device='cuda:0')
tensor([[0.6504],
        [0.5548],
        [0.6172],
        [0.5434]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.6012],
        [0.6783],
        [0.6093],
        [0.4088]], device='cuda:0')
tensor([[ True],
        [ True],
        [ True],
        [False]], device='cuda:0')
tensor([[0.5682],
        [0.5656],
        [0.4469],
        [0.5333]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [ True]], device='cuda:0')
tensor([[0.6006],
        [0.5605],
        [0.5082],
        [0.6639]], device='cuda:0')
tensor([[True],
        [True],
      